# Imports

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from PIL import Image

from uavloc.features.segmentation import CLASSES, UNET_ENCODER, load_unet
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.track import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

# Params

In [ ]:
TAG = "round3"
INIT_CKPT = "model-imagenet.pt"
BEFORE_CKPTS = {"satellite": "model-round2-bn-satellite.pt", "drone": "model-round2-bn-drone.pt"}
HOLDOUT_VILLAGE = "bushcha-50p2422-25p8305"
STRIP_VILLAGE = "birky-49p9105-23p9192"
STRIP_ROWS = 640
GAP_PX = 64
GOOGLE_DATE = "google"
DRONE_VAL_BLOCKS = [0, 4]
DRONE_P = 0.5
DRONE_GSD_M = 0.10
SAT_CROP_PX, SAT_BATCH = 512, 4
DRONE_CROP_PX, DRONE_BATCH = 256, 8
STEPS = 1500
EVAL_EVERY = 250
LR = 2e-4
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
RARE_CROP_P = 0.3
BN_BATCHES = 60
BN_SEED = 123
SEED = 0
SHOWCASE_COUNT = 6
SHOWCASE_WINDOW_PX = 1024

# Data

In [ ]:
DERIVED = REPO_ROOT / "data" / "derived"
VILLAGES = DERIVED / "ua-village-annotated"
DRONE = DERIVED / "ua-drone-annotated"
MULTIDATE = DERIVED / "ua-village-multidate"
HERE = REPO_ROOT / "pocs" / "village-seg-baseline"
FIGS = HERE / "figures"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(SEED)
DEVICE

In [ ]:
sat = vs.load_satellite(VILLAGES, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_VILLAGE, STRIP_VILLAGE, STRIP_ROWS, GAP_PX)
pd.DataFrame([{"location": s.loc.split("-")[0], "date": s.date, "train": not s.banned.all(),
               **{c: (s.lab == i).mean() for i, c in enumerate(CLASSES)}} for s in sat]).round(3)

In [ ]:
sat_train = [s for s in sat if not s.banned.all()]
sat_sampler = vs.SatelliteSampler(sat_train, SAT_CROP_PX, RARE_CROP_P)
{CLASSES[c]: len(p) for c, p in sat_sampler.centres.items()}

In [ ]:
drone = vs.load_drone(DRONE, DERIVED)
vs.split_by_footprint(drone, DRONE_VAL_BLOCKS)
round2_splits = json.loads((HERE / "results-round2.json").read_text())["splits"]
assert [d.name for d in drone if d.split == "train"] == round2_splits["drone_train"]
assert [d.name for d in drone if d.split == "val"] == round2_splits["drone_val"]
pd.Series([d.split for d in drone]).value_counts()

In [ ]:
drone_train = [d for d in drone if d.split == "train"]
drone_val = [d for d in drone if d.split == "val"]
drone_sampler = vs.DroneSampler(drone_train, DRONE_GSD_M, DRONE_CROP_PX, RARE_CROP_P)

In [ ]:
strip = [s for s in sat if s.loc == STRIP_VILLAGE]
VAL = {
    "birky-strip": [(s.img[:STRIP_ROWS], s.lab[:STRIP_ROWS]) for s in strip if s.date != GOOGLE_DATE],
    "birky-strip-google": [(s.img[:STRIP_ROWS], s.lab[:STRIP_ROWS]) for s in strip if s.date == GOOGLE_DATE],
    "bushcha": [(s.img, s.lab) for s in sat if s.loc == HOLDOUT_VILLAGE],
    "drone-val": [vs.drone_at_gsd(d, DRONE_GSD_M) for d in drone_val],
}
SAT_SETS = [n for n in VAL if n != "drone-val"]
{n: len(v) for n, v in VAL.items()}

In [ ]:
annotated = {s.loc for s in sat}
unseen_google = sorted(p.parent.name for p in MULTIDATE.glob("*/google.jpg") if p.parent.name not in annotated)
showcase = [unseen_google[i] for i in np.linspace(0, len(unseen_google) - 1, SHOWCASE_COUNT).astype(int)]
len(unseen_google), showcase

# Model

In [ ]:
def dice_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    ok = y != vs.IGNORE
    p = logits.softmax(1) * ok[:, None]
    onehot = F.one_hot(y.clamp(max=len(CLASSES) - 1), len(CLASSES)).permute(0, 3, 1, 2) * ok[:, None]
    present = onehot.sum((0, 2, 3)) > 0
    inter, den = (p * onehot).sum((0, 2, 3)), p.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    return (1 - (2 * inter + 1) / (den + 1))[present].mean()

In [ ]:
def bn_pair(model: torch.nn.Module) -> dict[str, torch.nn.Module]:
    """Same weights, BatchNorm statistics re-estimated per domain from its own training crops."""
    return {"satellite": vs.domain_bn(model, sat_sampler, SAT_BATCH, BN_BATCHES, BN_SEED, DEVICE),
            "drone": vs.domain_bn(model, drone_sampler, DRONE_BATCH, BN_BATCHES, BN_SEED, DEVICE)}

In [ ]:
def evaluate(models: dict[str, torch.nn.Module]) -> tuple[dict, dict]:
    """Satellite sets with the satellite-BN model, drone-val with the drone-BN model."""
    scores, preds = {}, {}
    for name, items in VAL.items():
        m = models["drone" if name == "drone-val" else "satellite"]
        preds[name] = [vs.predict(m, img, DEVICE) for img, _ in items]
        scores[name] = vs.iou_scores(preds[name], [lab for _, lab in items])
        scores[name]["agreement"] = vs.agreement(preds[name]) if name in SAT_SETS else None
    return scores, preds

In [ ]:
def miou_line(ev: dict) -> str:
    return "  ".join(f"{n} {ev[n]['miou']:.3f}" for n in VAL)

In [ ]:
before_models = {d: load_unet(HERE / c, DEVICE) for d, c in BEFORE_CKPTS.items()}
before, before_preds = evaluate(before_models)
miou_line(before)

In [ ]:
model = smp.Unet(UNET_ENCODER, encoder_weights=None, classes=len(CLASSES))
model.load_state_dict(torch.load(HERE / INIT_CKPT, map_location="cpu"))
model = model.to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=WARMUP_FRAC)

# Run

In [ ]:
rng = np.random.default_rng(SEED)
history, losses, t0 = [], {"sat": [], "drone": []}, time.time()
for step in range(1, STEPS + 1):
    model.train()
    domain = "drone" if rng.random() < DRONE_P else "sat"
    x, y = (vs.batch_of(drone_sampler, rng, DRONE_BATCH) if domain == "drone"
            else vs.batch_of(sat_sampler, rng, SAT_BATCH))
    out = model(x.to(DEVICE))
    loss = F.cross_entropy(out, y.to(DEVICE), ignore_index=vs.IGNORE) + dice_loss(out, y.to(DEVICE))
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    losses[domain].append(loss.item())
    if step % EVAL_EVERY == 0 or step == STEPS:
        ev, _ = evaluate(bn_pair(model))
        history.append({"step": step, "eval": ev, "loss": {k: float(np.mean(v)) for k, v in losses.items() if v}})
        print(f"step {step:4d}  {miou_line(ev)}  [{(time.time() - t0) / 60:.0f} min]", flush=True)
        losses = {"sat": [], "drone": []}
train_minutes = (time.time() - t0) / 60

In [ ]:
after_models = bn_pair(model)
torch.save(model.state_dict(), HERE / f"model-{TAG}.pt")
for domain, m in after_models.items():
    torch.save(m.state_dict(), HERE / f"model-{TAG}-bn-{domain}.pt")
after, after_preds = evaluate(after_models)
miou_line(after)

In [ ]:
results = {
    "tag": TAG, "init": INIT_CKPT, "before": BEFORE_CKPTS, "train_minutes": train_minutes,
    "params": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict))
               and k not in {"CLASSES", "VAL", "SAT_SETS"}},
    "splits": {"sat_train": [(s.loc, s.date) for s in sat_train], "drone_train": [d.name for d in drone_train],
               "drone_val": [d.name for d in drone_val], "showcase": showcase},
    "before_scores": before, "after_scores": after, "history": history,
}
(HERE / f"results-{TAG}.json").write_text(json.dumps(results, indent=1, default=str))

# Plots

In [ ]:
def colour(lab: np.ndarray) -> np.ndarray:
    out = np.full((*lab.shape, 3), 255, np.uint8)
    ok = lab != vs.IGNORE
    out[ok] = SEG_PALETTE[lab[ok]]
    return out

In [ ]:
rows = []
for name in VAL:
    for c, b in before[name]["iou"].items():
        rows.append({"set": name, "class": c, "round 2": b, "round 3": after[name]["iou"][c]})
    rows.append({"set": name, "class": "mIoU", "round 2": before[name]["miou"], "round 3": after[name]["miou"]})
    if before[name]["agreement"] is not None:
        rows.append({"set": name, "class": "agreement", "round 2": before[name]["agreement"],
                     "round 3": after[name]["agreement"]})
table = pd.DataFrame(rows).set_index(["set", "class"])
table.assign(delta=table["round 3"] - table["round 2"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, len(VAL), figsize=(5 * len(VAL), 3.6), sharey=True)
steps_ = [0] + [h["step"] for h in history]
for ax, name in zip(axes, VAL):
    ax.plot(steps_, [before[name]["miou"]] + [h["eval"][name]["miou"] for h in history], "k.-")
    ax.axhline(before[name]["miou"], color="0.6", ls="--", lw=1)
    ax.set_title(name); ax.set_xlabel("step (0 = round 2)"); ax.grid(alpha=.3)
axes[0].set_ylabel("mIoU, per-domain BN"); axes[0].set_ylim(0, 1)
plt.tight_layout(); plt.savefig(FIGS / f"{TAG}-curve.png", dpi=110); plt.show()

In [ ]:
legend = [plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))]
(img, lab), = VAL["birky-strip-google"]
panels = [(img, "Google (held-out strip)"), (colour(lab), "label"),
          (colour(before_preds["birky-strip-google"][0]), "round 2"),
          (colour(after_preds["birky-strip-google"][0]), "round 3")]
fig, axes = plt.subplots(len(panels), 1, figsize=(18, 3.8 * len(panels)))
for ax, (p, t) in zip(axes, panels):
    ax.imshow(p); ax.set_title(t); ax.axis("off")
fig.legend(legend, CLASSES, loc="lower center", ncol=len(CLASSES))
plt.tight_layout(rect=(0, .02, 1, 1)); plt.savefig(FIGS / f"{TAG}-birky-strip-google.png", dpi=80, bbox_inches="tight"); plt.show()

In [ ]:
def centre_window(arr: np.ndarray, size_px: int) -> np.ndarray:
    top, left = max((arr.shape[0] - size_px) // 2, 0), max((arr.shape[1] - size_px) // 2, 0)
    return arr[top:top + size_px, left:left + size_px]

In [ ]:
showcase_preds = {}
for loc in showcase:
    google = np.array(Image.open(MULTIDATE / loc / "google.jpg").convert("RGB"))
    esri = np.array(Image.open(MULTIDATE / loc / "current.jpg").convert("RGB"))
    showcase_preds[loc] = {"google": google, "esri": esri,
                           "round 2 on Google": vs.predict(before_models["satellite"], google, DEVICE),
                           "round 3 on Google": vs.predict(after_models["satellite"], google, DEVICE),
                           "round 3 on Esri": vs.predict(after_models["satellite"], esri, DEVICE)}

In [ ]:
cols = ["google", "round 2 on Google", "round 3 on Google", "esri", "round 3 on Esri"]
fig, axes = plt.subplots(len(showcase), len(cols), figsize=(4.2 * len(cols), 4.3 * len(showcase)), squeeze=False)
for r, loc in enumerate(showcase):
    for ax, col in zip(axes[r], cols):
        p = showcase_preds[loc][col]
        ax.imshow(centre_window(p if p.ndim == 3 else colour(p), SHOWCASE_WINDOW_PX)); ax.set_xticks([]); ax.set_yticks([])
        if r == 0: ax.set_title(col)
    axes[r, 0].set_ylabel(loc.rsplit("-", 2)[0])
fig.legend(legend, CLASSES, loc="lower center", ncol=len(CLASSES))
plt.tight_layout(rect=(0, .015, 1, 1)); plt.savefig(FIGS / f"{TAG}-google-unseen.png", dpi=70, bbox_inches="tight"); plt.show()

In [ ]:
shares = pd.DataFrame([{"village": loc.rsplit("-", 2)[0], "model / source": col,
                        **{c: (showcase_preds[loc][col] == i).mean() for i, c in enumerate(CLASSES) if c != "field"}}
                       for loc in showcase for col in cols if showcase_preds[loc][col].ndim == 2])
shares.set_index(["village", "model / source"]).round(3)

In [ ]:
results["showcase_class_shares"] = shares.to_dict("records")
(HERE / f"results-{TAG}.json").write_text(json.dumps(results, indent=1, default=str))